In [ ]:
# This example will demonstrate:

    # Defining a Custom Tool: How to create a Python class that encapsulates the logic for interacting with your MCP server.
    # Creating Agents: How to define AI agents with specific roles, goals, and backstories.
    # Defining Tasks: How to set up tasks that your agents will perform, utilizing the custom tools.
    # Orchestrating a Crew: How to bring agents and tasks together into a collaborative crew.

# --- 0. Import CrewAI ---

In [1]:
import os 
# This line imports the os module, which provides a way of using operating system-dependent functionality. 
# One of its common uses is to interact with environment variables.

from crewai import Agent, Task, Crew, LLM
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
from typing import Type
from linkup import LinkupClient

from dotenv import load_dotenv
# Load environment variables from a .env file if it exists.
# This is a good practice for managing API keys and other secrets.
load_dotenv()


True

# --- 0. Import Opik ---

In [2]:
# Import opik and its CrewAI integration
import opik
from opik.integrations.crewai import track_crewai
track_crewai(project_name="arunmanglick-crewai-integration-demo")

# --- 1. Integrate Linkup Search Tool ---

In [3]:
class LinkUpSearchInput(BaseModel):
    """Input schema for LinkUp Search Tool."""
    query: str = Field(description="The search query to perform")
    depth: str = Field(default="standard",
                       description="Depth of search: 'standard' or 'deep'")
    output_type: str = Field(
        default="searchResults", description="Output type: 'searchResults', 'sourcedAnswer', or 'structured'")
    
# Code Explanation
# The code is a Pydantic data model that defines the input parameters for the LinkUpSearchTool. 
# It acts as a contract, telling the AI agent exactly what information it needs to provide when it decides to use this tool.

# Here's a breakdown of what the code does:

# class LinkUpSearchInput(BaseModel):: This line defines a new class that inherits from BaseModel. 
# Pydantic's BaseModel is used to create data models that automatically validate data types and provide a structured way to handle inputs. 
# In this case, it ensures the LinkUpSearchTool always receives the correct arguments in the correct format.

    # query: str = Field(...): This defines the main search query.
    # query: str specifies that the query field must be a string.
    # Field(description="...") is a Pydantic function that adds metadata to the field. 
    # The description is crucial because it helps the LLM understand what to put in this field. 
    # It's an essential part of how the agent knows what to do.

    # depth: str = Field(...): This defines the depth of the search.
    # depth: str again enforces that the input must be a string.
    # Field(default="standard", ...) sets a default value of "standard". 
    # This means the agent doesn't have to specify this parameter every time; it will automatically use the default unless 
    # it has a reason to change it to "deep".

    # output_type: str = Field(...): This defines the type of output the tool should return.
    # output_type: str ensures the input is a string.
    # Field(default="searchResults", ...) sets a default value, similar to the depth parameter. 
    # The agent will get searchResults by default but has the option to request sourcedAnswer or structured output if its task requires it.

# In short, this Pydantic model provides a clear, documented, and validated schema for the LinkUpSearchTool,
# making it easy for the AI agent to use the tool correctly and for developers to understand its functionality.


In [4]:
class LinkUpSearchTool(BaseTool):
    name: str = "LinkUp Search"
    description: str = "Search the web for information using LinkUp and return comprehensive results"
    args_schema: Type[BaseModel] = LinkUpSearchInput

    def __init__(self):
        super().__init__()

    def _run(self, query: str, depth: str = "standard", output_type: str = "searchResults") -> str:
        """Execute LinkUp search and return results."""
        try:
            # Initialize LinkUp client with API key from environment variables
            linkup_client = LinkupClient(api_key=os.getenv("LINKUP_API_KEY"))

            # Perform search
            search_response = linkup_client.search(
                query=query,
                depth=depth,
                output_type=output_type
            )

            return str(search_response)
        except Exception as e:
            return f"Error occurred while searching: {str(e)}"

# --- 1. Setup Local LLM ---

In [5]:
# The LLM class is used to configure a language model.
# We are specifying 'ollama/llama3.2' as the model and pointing to the default local Ollama server address.

local_llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)

# What is Ollama
# Ollama is a platform that lets you run LLMs locally on your machine—no cloud dependency required. 
# It’s designed for developers who want fast, private, and customizable access to models like LLaMA, Mistral, Gemma, Phi-4, and more.

# --- 2. Define AI Agent ---

In [6]:
# An Agent has a specific role, goal, and backstory.
# It also uses the LLM we just configured.
my_research_aiagent = Agent(
     role="Web Search Agent",
    goal="Search the web for information",
    backstory="You are an agent that can search the web for information.",
    verbose=True,
    allow_delegation=False,
    tools=[LinkUpSearchTool()],  # The agent now has the LinkUpSearchTool available
    llm=local_llm  # Assign the local LLM to this agent
)

# Code Explaination

# role='AI Researcher': This gives the agent its professional identity. 
# The role helps the agent understand its purpose and the type of actions it should take.

# goal='Provide a concise summary of the benefits of using local LLMs':
# This is the agent's main objective. It's the specific task the agent is trying to accomplish within the crew's workflow.

# backstory=(...): 
# This provides context and personality. The backstory helps the agent adopt a specific persona and knowledge base, 
# influencing how it approaches and solves its tasks. In this case, the agent is an "expert AI researcher."

# verbose=True: When set to True, this parameter makes the agent's internal thought process visible. 
# You'll see the agent's reasoning, tool usage, and progress as it works, which is very useful for debugging and 
# understanding its behavior.

# allow_delegation=False: This setting determines whether the agent can pass a task on to another agent in the crew. 
# Here, False means this agent must handle all tasks itself without delegating.


# --- 3. Define the Task ---

In [7]:
# A Task is a specific piece of work for an agent.
# It includes a description, the expected output, and the agent to whom it's assigned.

# research_task = Task(
#     description=(
#         "Research and list at least three key benefits of using a local LLM "
#         "like Ollama over a cloud-based service. "
#         "Focus on aspects like data privacy, cost, and customization. "
#         "The final output should be a clear, concise, and easy-to-read summary."
#     ),
#     expected_output="A summary listing at least three benefits of local LLMs.",
#     agent=my_research_aiagent
# )

research_task = Task(
    description="Research and provide answer to the user's {query} ",
    expected_output="A summary to the user's query",
    agent=my_research_aiagent
)

# --- 4. Create the Crew ---

In [8]:
# A Crew is a collection of agents and their assigned tasks.
# We're using a simple sequential process here, meaning tasks are executed one after the other.

ai_crew = Crew(
    agents=[my_research_aiagent],
    tasks=[research_task],
    verbose=True  # Set verbosity to 2 to see all the agent's thoughts and actions
)

# --- 5. Kick off the crew's work ---

In [ ]:
import sys

print("--- Crew is starting its work ---")
result = ai_crew.kickoff({"query": "What is latest news on AI from Sam Altman?"})

print("\n\n--- Crew's final output ---")
print(result)

# Graceful shutdown
sys.stdout.flush()
os._exit(0)

--- Crew is starting its work ---


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 767eaf09-3c00-409e-bdd9-30b485fcc87e                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Search Agent                                                                                        │
│                                                                                                                 │
│  Task: Research and provide answer to the user's What is latest news on AI from Sam Altman?                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Search Agent                                                                                        │
│                                                                                                                 │
│  Thought: Action: LinkUp Search                                                                                 │
│                                                                                                                 │
│  Using Tool: LinkUp Search                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"query\": \"latest news on AI from Sam Altman\", \"depth\": \"standard\", \"output_type\":                  │
│  \"searchResults\"}"                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  results=[LinkupSearchTextResult(type='text', name='Sam Altman', url='https://blog.samaltman.com/',             │
│  content='From here on, the tools we have already built will help us find further scientific insights and aid   │
│  us in creating better AI systems. Of course this isn’t the same thing as an AI system completely autonomously  │
│  updating its own code, but nevertheless this is a larval version of recursive ...\nFrom here on, the tools we  │
│  have already built will help us find further scientific insights and aid us in creating better AI systems. Of  │
│  course this isn’t the same thing as an AI system completely autonomously updating its own code, but            │
│  nevertheless this is a larval version of recursive self-improvement.\nIt’s hard to even imagine today what we  │
│  will have discovered by 2035; maybe we will go from solving high-energy physics one year to beginning space    │
│  colonization the next year; or from a major materials science breakthrough one year to true high-bandwidth     │
│  brain-computer interfaces the next year. Many people will choose to live their lives in much the same way,     │
│  but at least some people will probably decide to “plug in”.\nWe offer this for code and now image generation;  │
│  both of these will get a lot better. But the same trend will happen in new ways until eventually it works for  │
│  complex tasks—we can imagine an “AI office worker” that takes requests in natural language like a human        │
│  does.\nThen everyone else agrees, YC founders and otherwise (non-YC founders might talk about an impactful     │
│  essay or getting hired at a YC company). Jessica still sadly doesn’t get nearly the same degree of public      │
│  credit, but the people who were around the early days of YC know the real story.'),                            │
│  LinkupSearchTextResult(type='text', name="OpenAI's Sam Altman sees AI bubble forming as industry spending      │
│  surges", url='https://www.cnbc.com/2025/08/18/openai-sam-altman-warns-ai-market-is-in-a-bubble.html',          │
│  content='OpenAI CEO Sam Altman has reportedly said that he believes AI could be in a bubble, compa...          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()